# Project Vesta — Operator-Spectral Transformer Attention Experiment

This notebook reproduces the initial Project Vesta experiment.

It:

1. Loads DistilGPT2  
2. Extracts transformer attention matrices  
3. Constructs symmetric surrogate operators  
4. Computes eigenvalues and spectral gaps  
5. Generates layerwise plots and attention-head heatmaps  
6. Compares coherent vs corrupted prompts  

This is exploratory research. It does **not** claim that transformers are quantum systems or that spectral gaps directly encode semantic truth.


## 1. Install dependencies

In [ ]:
!pip install transformers torch matplotlib numpy -q

## 2. Import libraries

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM


## 3. Load DistilGPT2

In [ ]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_attentions=True
)

model.eval()


## 4. Single-prompt attention extraction

In [ ]:
prompt = "2 + 3 ="

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions

A = attentions[0][0, 0].detach().numpy()

print("Attention matrix shape:", A.shape)
print(A)


## 5. Construct symmetric surrogate operator

Given an attention matrix A, define:

H = 1/2(A + A^T)

This makes the operator symmetric, giving real eigenvalues.


In [ ]:
H = 0.5 * (A + A.T)

eigenvalues = np.linalg.eigvalsh(H)

largest = eigenvalues[-1]
second = eigenvalues[-2]
spectral_gap = largest - second

print("Eigenvalues:", eigenvalues)
print("Largest eigenvalue:", largest)
print("Spectral gap:", spectral_gap)


## 6. Plot eigenvalue spectrum

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(eigenvalues, marker="o")
plt.title("Eigenvalue Spectrum")
plt.xlabel("Index")
plt.ylabel("Eigenvalue")
plt.grid(True)
plt.show()


## 7. Compare prompts by spectral gap

In [ ]:
prompts = [
    "2 + 3 =",
    "7 + 8 =",
    "12 - 5 =",
    "The capital of France is",
    "The moon is made of",
    "If all bloops are razzies and all razzies are lazzies,"
]

results = []

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    attentions = outputs.attentions

    A = attentions[0][0, 0].detach().numpy()
    H = 0.5 * (A + A.T)

    eigenvalues = np.linalg.eigvalsh(H)
    largest = eigenvalues[-1]
    second = eigenvalues[-2]
    spectral_gap = largest - second

    results.append({
        "prompt": prompt,
        "largest_eigenvalue": float(largest),
        "spectral_gap": float(spectral_gap)
    })

for r in results:
    print("\n-------------------")
    print("Prompt:", r["prompt"])
    print("Largest Eigenvalue:", r["largest_eigenvalue"])
    print("Spectral Gap:", r["spectral_gap"])


## 8. Semantic perturbation test

In [ ]:
prompts = [
    "The capital of France is Paris",
    "The capital of France is banana",
    "The capital of Japan is Tokyo",
    "The capital of Japan is sandwich",
    "The capital of Italy is Rome",
    "The capital of Italy is guitar",
    "The capital of Germany is Berlin",
    "The capital of Germany is cloud",
]

results = []

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    attentions = outputs.attentions

    A = attentions[0][0, 0].detach().numpy()
    H = 0.5 * (A + A.T)

    eigenvalues = np.linalg.eigvalsh(H)
    largest = eigenvalues[-1]
    second = eigenvalues[-2]
    spectral_gap = largest - second

    results.append({
        "prompt": prompt,
        "largest_eigenvalue": float(largest),
        "spectral_gap": float(spectral_gap)
    })

for r in results:
    print("\n-------------------")
    print("Prompt:", r["prompt"])
    print("Largest Eigenvalue:", r["largest_eigenvalue"])
    print("Spectral Gap:", r["spectral_gap"])


## 9. Layerwise spectral gap evolution

In [ ]:
prompt = "The capital of France is Paris"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions

mean_gaps = []

for layer in range(len(attentions)):
    gaps = []

    for head in range(attentions[layer].shape[1]):
        A = attentions[layer][0, head].detach().numpy()
        H = 0.5 * (A + A.T)

        eigenvalues = np.linalg.eigvalsh(H)
        gap = eigenvalues[-1] - eigenvalues[-2]

        gaps.append(float(gap))

    mean_gaps.append(np.mean(gaps))

plt.figure(figsize=(8, 5))
plt.plot(mean_gaps, marker="o")
plt.title("Mean Spectral Gap Across Transformer Layers")
plt.xlabel("Layer")
plt.ylabel("Mean Spectral Gap")
plt.grid(True)
plt.show()

print("Mean gaps by layer:")
print(mean_gaps)


## 10. Spectral gap heatmap across layers and heads

In [ ]:
prompt = "The capital of France is Paris"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions

gap_matrix = []

for layer in range(len(attentions)):
    layer_gaps = []

    for head in range(attentions[layer].shape[1]):
        A = attentions[layer][0, head].detach().numpy()
        H = 0.5 * (A + A.T)

        eigenvalues = np.linalg.eigvalsh(H)
        gap = eigenvalues[-1] - eigenvalues[-2]

        layer_gaps.append(float(gap))

    gap_matrix.append(layer_gaps)

gap_matrix = np.array(gap_matrix)

plt.figure(figsize=(10, 6))
plt.imshow(gap_matrix, aspect="auto")
plt.colorbar(label="Spectral Gap")
plt.xlabel("Attention Head")
plt.ylabel("Layer")
plt.title("Spectral Gap Heatmap Across Layers and Heads")
plt.show()


## 11. Semantic perturbation heatmap

In [ ]:
prompt1 = "The capital of France is Paris"
prompt2 = "The capital of France is banana"

def compute_gap_matrix(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    attentions = outputs.attentions

    gap_matrix = []

    for layer in range(len(attentions)):
        layer_gaps = []

        for head in range(attentions[layer].shape[1]):
            A = attentions[layer][0, head].detach().numpy()
            H = 0.5 * (A + A.T)

            eigenvalues = np.linalg.eigvalsh(H)
            gap = eigenvalues[-1] - eigenvalues[-2]

            layer_gaps.append(float(gap))

        gap_matrix.append(layer_gaps)

    return np.array(gap_matrix)

G1 = compute_gap_matrix(prompt1)
G2 = compute_gap_matrix(prompt2)

difference = G1 - G2

plt.figure(figsize=(10, 6))
plt.imshow(difference, aspect="auto", cmap="coolwarm")
plt.colorbar(label="Gap Difference")
plt.xlabel("Attention Head")
plt.ylabel("Layer")
plt.title("Spectral Gap Difference: Paris - Banana")
plt.show()


## 12. Notes and limitations

Current limitations:

- Only DistilGPT2 is tested.
- Prompt sample size is small.
- Symmetrizing attention discards directional structure.
- Spectral gap interpretation is not yet causally established.
- Results should be treated as exploratory diagnostics, not proof of semantic understanding.

Next steps:

- Expand prompt dataset.
- Test multiple transformer models.
- Compare against randomized attention baselines.
- Add entropy, eigenvector overlap, and norm-growth metrics.
- Evaluate robustness across prompt categories.
